In [1]:
# inlegalbert_kg_rag_scl_rrc.py
#
# Architecture:
#   InLegalBERT → BiLSTM → Multi-Head Attention Pooling → Linear → CRF
#   + Knowledge Graph (KG) with RST-based edges
#   + Uncertainty-triggered KG Retrieval + Graph Attention Fusion
#   + Supervised Contrastive Learning (SCL) for minority class handling  [NEW]
#
# WHAT IS NEW vs inlegalbert_kg_rag_rrc.py:
# ─────────────────────────────────────────
#  [SCL-1] SupConLoss:
#       A supervised contrastive loss added at the BiLSTM (sent_out_dim=256)
#       embedding level.  For every anchor sentence, positives = same label,
#       negatives = different labels.  Minority classes are upweighted via
#       per-class inverse-frequency weights so the model prioritises pulling
#       rare-class embeddings together tightly even though they are few.
#
#  [SCL-2] Class-balanced contrastive sampling (BalancedContrastiveSampler):
#       Within each document batch, we OVERSAMPLE minority-class sentences
#       so that the contrastive loss sees a balanced set of anchors.
#       Without this, majority classes dominate the loss computation and
#       minority classes get almost no gradient signal from SCL.
#
#  [SCL-3] Dual-loss in Phase A (base training):
#       total_loss = CRF_loss + AUX_CE_WEIGHT * CE_loss
#                  + SCL_WEIGHT * SupConLoss          [NEW]
#       The SCL acts on sent_vecs (before ctx_bilstm), so it directly
#       shapes the embedding space that the KG will later index.
#
#  [SCL-4] Dual-loss in Phase B (KG-augmented fine-tuning):
#       total_loss = (base_CRF + fused_CRF)/2 + AUX_CE * CE
#                  + SCL_WEIGHT * SupConLoss(fused_sent_vecs)  [NEW]
#       After KG fusion, fused embeddings are also pulled into tight
#       per-class clusters, making KG retrieval more discriminative.
#
#  WHY SCL SPECIFICALLY HELPS MINORITY CLASSES:
#  ─────────────────────────────────────────────
#  • Standard CE optimises per-sample softmax — it does NOT control
#    how embeddings are arranged in representation space.
#  • Minority classes end up scattered and close to majority-class
#    clusters because they are underrepresented in CE gradients.
#  • SCL with per-class inverse-frequency weights multiplies the
#    contrastive gradient for minority classes, forcing their embeddings
#    to form tight, well-separated clusters even with few samples.
#  • This improved representation directly benefits KG retrieval:
#    the KG node embeddings become more discriminative for rare labels,
#    so KG context retrieved for uncertain/rare sentences is more
#    label-relevant.
#  • Proven in: Khosla et al. 2020 (SupCon), Zhu et al. 2022
#    (long-tail SCL), and empirically in legal NLP imbalance settings.
#
# All original KG-RAG components are UNCHANGED.
# ──────────────────────────────────────────────────────────────────────

import os, json, random, time, math
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_rag_scl_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 20
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2

MHA_HEADS    = 4
MHA_DROPOUT  = 0.1

CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1

ES_PATIENCE  = 10
ES_MIN_DELTA = 1e-4

WARMUP_RATIO = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD = 0.05

# ── KG-RAG specific ────────────────────────────────────────
KG_TOP_K            = 3
KG_TOP_NODES        = 5
KG_HOP              = 1
UNCERTAINTY_THRESH  = 0.7
RARE_ALWAYS_KG      = True
KG_FUSION_DIM       = 256   # = SENT_LSTM_HIDDEN * 2

RST_INTRA_THRESH    = 0.6
RST_CROSS_THRESH    = 0.5

# ── SCL specific [NEW] ─────────────────────────────────────
SCL_WEIGHT          = 0.3    # weight of SupCon loss in total loss
SCL_TEMPERATURE     = 0.07   # temperature τ in contrastive loss (lower = sharper)
SCL_MINORITY_BOOST  = 2.0    # extra multiplier on minority-class contrastive weight
# Min positives required to compute SCL for an anchor (skip if fewer)
SCL_MIN_POSITIVES   = 1
# Balanced sampling: how many sentences to sample per class for SCL
SCL_SAMPLES_PER_CLASS = 4

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# [SCL-1] SUPERVISED CONTRASTIVE LOSS
# ═══════════════════════════════════════════════════════════
class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al., 2020) extended with
    per-class inverse-frequency weighting for minority class handling.

    Given N sentence embeddings z_i with labels y_i:
      L = -1/|A| Σ_{i∈A} [w_i / |P(i)|  Σ_{p∈P(i)} log
              exp(z_i·z_p / τ)
              ─────────────────────────────────────────]
              Σ_{a∈A-{i}} exp(z_i·z_a / τ)

    where:
      P(i)  = set of indices with same label as i (positives)
      A     = all indices (anchors + negatives)
      w_i   = class weight for label y_i (inverse freq, boosted for minority)
      τ     = temperature (SCL_TEMPERATURE)

    Why this beats standard CE for minority classes:
    ─────────────────────────────────────────────────
    • CE loss gradient ∝ (prediction_error).  For minority classes,
      there are few samples so total gradient mass is small.
    • SCL gradient depends on PAIRWISE similarity — every minority
      sample is compared against ALL other samples. With w_i boost,
      the minority gradient signal is amplified relative to majority.
    • Result: minority class embeddings form tight, separable clusters
      even when sample count is low.
    """
    def __init__(
        self,
        temperature:     float = SCL_TEMPERATURE,
        base_temperature: float = 0.07,
        minority_boost:  float = SCL_MINORITY_BOOST,
    ):
        super().__init__()
        self.temperature      = temperature
        self.base_temperature = base_temperature
        self.minority_boost   = minority_boost

    def forward(
        self,
        features:      torch.Tensor,   # (N, D) L2-normalised embeddings
        labels:        torch.Tensor,   # (N,)   integer class ids
        class_weights: torch.Tensor,   # (C,)   per-class weights (inv freq)
        rare_ids:      list = None,
    ) -> torch.Tensor:
        """
        Args:
            features:      L2-normalised sentence embeddings (N, D)
            labels:        ground-truth label ids (N,)
            class_weights: per-class inverse-frequency weights (C,)
            rare_ids:      list of minority class ids for extra boost
        Returns:
            scalar SCL loss
        """
        device = features.device
        N = features.shape[0]

        if N < 2:
            return torch.tensor(0.0, device=device, requires_grad=True)

        # ── Build per-sample weights ───────────────────
        # w_i = class_weight[y_i], with extra boost for minority classes
        sample_weights = class_weights[labels]               # (N,)
        if rare_ids:
            rare_mask = torch.zeros(N, dtype=torch.bool, device=device)
            for rid in rare_ids:
                rare_mask |= (labels == rid)
            sample_weights = sample_weights.clone()
            sample_weights[rare_mask] *= self.minority_boost

        # ── Cosine similarity matrix ───────────────────
        # features already L2-normalised → dot product = cosine sim
        sim_matrix = torch.mm(features, features.T) / self.temperature  # (N, N)

        # ── Positive and negative masks ────────────────
        labels_col = labels.unsqueeze(0)    # (1, N)
        labels_row = labels.unsqueeze(1)    # (N, 1)
        pos_mask   = (labels_row == labels_col).float()   # (N, N)
        self_mask  = torch.eye(N, device=device)
        pos_mask   = pos_mask - self_mask   # remove self-pair from positives

        # ── Check: at least one positive per anchor ────
        n_pos_per_anchor = pos_mask.sum(dim=1)   # (N,)
        valid = n_pos_per_anchor >= SCL_MIN_POSITIVES

        if not valid.any():
            return torch.tensor(0.0, device=device, requires_grad=True)

        # ── Log-softmax denominator (all pairs except self) ──
        # For numerical stability: subtract max before exp
        logits_mask  = 1.0 - self_mask
        exp_sim      = torch.exp(
            sim_matrix - sim_matrix.max(dim=1, keepdim=True).values
        ) * logits_mask  # (N, N)

        log_prob = sim_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-9)

        # ── Per-anchor loss ────────────────────────────
        # Mean log-prob over positives, weighted by sample_weights
        mean_log_prob_pos = (pos_mask * log_prob).sum(dim=1) / (
            n_pos_per_anchor.clamp(min=1)
        )  # (N,)

        # Apply per-sample class weight
        loss_per_anchor = -(self.temperature / self.base_temperature) \
                          * mean_log_prob_pos * sample_weights

        # Only average over valid anchors (those with ≥1 positive)
        loss = loss_per_anchor[valid].mean()
        return loss


# ═══════════════════════════════════════════════════════════
# [SCL-2] BALANCED CONTRASTIVE SAMPLER
# ═══════════════════════════════════════════════════════════
class BalancedContrastiveSampler:
    """
    For each document batch, collect all (embedding, label) pairs and
    return a class-balanced subset for SCL computation.

    Without balancing, majority classes dominate the (anchor, positive)
    pairs and minority classes receive negligible gradient from SCL.

    Strategy:
      - For each class present in the batch, keep up to
        SCL_SAMPLES_PER_CLASS sentences.
      - For minority classes with fewer than SCL_SAMPLES_PER_CLASS
        sentences, REPEAT (oversample) existing embeddings with small
        Gaussian noise to create synthetic positives in embedding space.
        This is feature-level oversampling (safer than SMOTE on raw text).
    """
    def __init__(
        self,
        samples_per_class: int   = SCL_SAMPLES_PER_CLASS,
        noise_std:         float = 0.01,
        rare_ids:          list  = None,
    ):
        self.samples_per_class = samples_per_class
        self.noise_std         = noise_std
        self.rare_ids          = rare_ids or []

    def sample(
        self,
        embeddings: torch.Tensor,   # (N, D)
        labels:     torch.Tensor,   # (N,)
    ):
        """
        Returns:
            sampled_embs   (M, D)  — balanced embeddings
            sampled_labels (M,)    — corresponding labels
        """
        device    = embeddings.device
        emb_list  = []
        lbl_list  = []

        unique_labels = labels.unique().tolist()

        for lbl in unique_labels:
            idx  = (labels == lbl).nonzero(as_tuple=True)[0]
            embs = embeddings[idx]    # (k, D)
            k    = embs.shape[0]

            if k >= self.samples_per_class:
                # Randomly subsample
                chosen = torch.randperm(k, device=device)[:self.samples_per_class]
                emb_list.append(embs[chosen])
            else:
                # Keep all real samples
                emb_list.append(embs)
                # Oversample with noise to reach samples_per_class
                # (only for minority/rare classes)
                n_needed = self.samples_per_class - k
                if int(lbl) in self.rare_ids or k < self.samples_per_class:
                    repeats = embs[torch.randint(0, k, (n_needed,), device=device)]
                    noise   = torch.randn_like(repeats) * self.noise_std
                    emb_list.append(repeats + noise)
                    k = self.samples_per_class  # updated count

            n_added = min(k, self.samples_per_class)
            lbl_list.append(
                torch.full((n_added,), lbl, dtype=torch.long, device=device)
            )

        if not emb_list:
            return embeddings, labels

        sampled_embs   = torch.cat(emb_list, dim=0)
        sampled_labels = torch.cat(lbl_list, dim=0)

        # Shuffle
        perm = torch.randperm(sampled_embs.shape[0], device=device)
        return sampled_embs[perm], sampled_labels[perm]


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


def compute_class_weights(label_freqs: dict, rare_ids: list,
                           smoothing: float = 0.1) -> torch.Tensor:
    """
    Compute inverse-frequency class weights for SCL.
    w_c = 1 / (freq_c + smoothing), then normalise to mean=1.

    Returns Tensor (NUM_LABELS,) on CPU.
    """
    weights = []
    for i in range(NUM_LABELS):
        freq = label_freqs.get(id2label[i], 0.0)
        weights.append(1.0 / (freq + smoothing))
    weights = torch.tensor(weights, dtype=torch.float)
    weights = weights / weights.mean()   # normalise so mean weight = 1
    print("\n⚖️  SCL class weights (inverse freq, normalised):")
    for i, lbl in enumerate(LABELS):
        flag = " ← RARE" if label2id[lbl] in rare_ids else ""
        print(f"   {lbl:<20}  weight={weights[i]:.3f}{flag}")
    return weights


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING  (unchanged)
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)

        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL  (extended with SCL projection head)
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):
    """
    Original base model with one addition:
      self.scl_proj  — a small 2-layer MLP projection head that maps
                       sent_vecs (sent_out_dim=256) to a normalised
                       contrastive embedding space (scl_proj_dim).
                       This head is used ONLY during training for SCL;
                       it is discarded at inference (standard practice
                       from SimCLR / SupCon).
    """
    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        scl_proj_dim     = 128,          # [SCL] projection head output dim
    ):
        super().__init__()

        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size    # 768
        self.dropout  = nn.Dropout(dropout)

        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2        # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = self.sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = self.sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2          # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )

        self.crf = CRF(num_tags=num_labels, batch_first=True)

        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100,
        )

        # ── [SCL] Projection head ──────────────────────
        # Maps sent_out_dim → scl_proj_dim (normalised)
        # Used only during training; NOT used in decode path.
        self.scl_proj = nn.Sequential(
            nn.Linear(self.sent_out_dim, self.sent_out_dim),
            nn.GELU(),
            nn.Linear(self.sent_out_dim, scl_proj_dim),
        )
        self.scl_proj_dim = scl_proj_dim

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total   = len(encoder_layers)
        n_trained = n_total - n_freeze
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} + pooler.\n")

    def encode_sentences(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ) -> torch.Tensor:
        B, T, L = input_ids.shape
        N = B * T

        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)

        token_embs_all = self.dropout(token_embs_all)

        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)

        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)

        return sent_vecs.view(B, T, -1)

    def get_scl_embeddings(
        self,
        sent_vecs: torch.Tensor,   # (B, T, sent_out_dim)
        labels:    torch.Tensor,   # (B, T)
    ):
        """
        [SCL] Flatten, apply projection head, L2-normalise.
        Returns (embeddings (N, scl_proj_dim), flat_labels (N,))
        where N = sum of valid (non-padding) sentences.
        """
        B, T, D = sent_vecs.shape
        flat_vecs   = sent_vecs.reshape(B * T, D)          # (B*T, D)
        flat_labels = labels.reshape(B * T)                 # (B*T,)

        # Keep only valid (non-padding) sentences
        valid_mask = flat_labels != -100
        flat_vecs   = flat_vecs[valid_mask]                 # (N, D)
        flat_labels = flat_labels[valid_mask]               # (N,)

        if flat_vecs.shape[0] == 0:
            return None, None

        proj = self.scl_proj(flat_vecs)                     # (N, scl_proj_dim)
        proj = F.normalize(proj, dim=-1)                    # L2 normalise
        return proj, flat_labels

    def get_emissions(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs_drop = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
        class_weights:  torch.Tensor = None,   # [SCL] per-class weights
        scl_loss_fn:    nn.Module    = None,   # [SCL] SupConLoss instance
        rare_ids:       list         = None,   # [SCL] minority class ids
        scl_sampler:    object       = None,   # [SCL] BalancedContrastiveSampler
    ):
        _, _, emissions = self.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")

            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
            )

            # ── [SCL-3] Compute SCL on sent_vecs ──────────
            scl_loss = torch.tensor(0.0, device=emissions.device)
            if scl_loss_fn is not None and class_weights is not None:
                # Need sent_vecs — re-encode (or get from get_emissions)
                sent_vecs, _, _ = self.get_emissions(
                    input_ids, attention_mask, token_type_ids, lengths=lengths
                )
                proj_embs, flat_labels = self.get_scl_embeddings(sent_vecs, labels)
                if proj_embs is not None and proj_embs.shape[0] >= 2:
                    # Class-balanced sampling [SCL-2]
                    if scl_sampler is not None:
                        proj_embs, flat_labels = scl_sampler.sample(
                            proj_embs, flat_labels
                        )
                    cw = class_weights.to(emissions.device)
                    scl_loss = scl_loss_fn(
                        proj_embs, flat_labels, cw, rare_ids=rare_ids
                    )

            loss = (crf_loss
                    + AUX_CE_WEIGHT * ce_loss
                    + SCL_WEIGHT    * scl_loss)     # [SCL] added term

            return loss, emissions, scl_loss.item()
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# KNOWLEDGE GRAPH  (unchanged)
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    def __init__(self, emb_dim=KG_FUSION_DIM):
        self.emb_dim     = emb_dim
        self.nodes       = defaultdict(list)
        self.intra_edges = defaultdict(list)
        self.cross_edges = []
        self._stacked    = {}

    def add_nodes(self, embeddings: torch.Tensor, label_ids: list, texts: list = None):
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            text = texts[k] if texts is not None else ""
            self.nodes[lid].append({"emb": emb, "text": text})
        self._stacked = {}

    def build_edges(self,
                    intra_thresh=RST_INTRA_THRESH,
                    cross_thresh=RST_CROSS_THRESH,
                    max_intra_edges_per_node=5,
                    max_cross_edges=2000):
        print("  Building KG edges ...")
        self._stacked = {}
        self.intra_edges = defaultdict(list)
        self.cross_edges = []

        for lid, node_list in self.nodes.items():
            N = len(node_list)
            if N < 2:
                continue
            embs = torch.stack([n["emb"] for n in node_list])
            embs_norm = F.normalize(embs, dim=-1)
            sim_mat = torch.mm(embs_norm, embs_norm.T)

            for i in range(N - 1):
                w = float(sim_mat[i, i + 1].item())
                self.intra_edges[lid].append((i, i + 1, max(0.0, w)))

            for i in range(N):
                sims = sim_mat[i].clone()
                sims[max(0, i-1):i+2] = -1
                count = 0
                while count < max_intra_edges_per_node:
                    j = int(sims.argmax().item())
                    if sims[j] < intra_thresh:
                        break
                    w = float(sims[j].item())
                    self.intra_edges[lid].append((i, j, w))
                    sims[j] = -1
                    count += 1

            self._stacked[lid] = embs

        label_ids = list(self.nodes.keys())
        cross_count = 0
        for a in range(len(label_ids)):
            if cross_count >= max_cross_edges:
                break
            for b in range(a + 1, len(label_ids)):
                if cross_count >= max_cross_edges:
                    break
                la, lb = label_ids[a], label_ids[b]
                embs_a = self._get_stacked(la)
                embs_b = self._get_stacked(lb)
                if embs_a is None or embs_b is None:
                    continue
                na_norm = F.normalize(embs_a, dim=-1)
                nb_norm = F.normalize(embs_b, dim=-1)
                sim_mat = torch.mm(na_norm, nb_norm.T)
                high = (sim_mat >= cross_thresh).nonzero(as_tuple=False)
                for pair in high[:50]:
                    ni, nj = int(pair[0]), int(pair[1])
                    w = float(sim_mat[ni, nj].item())
                    self.cross_edges.append((la, ni, lb, nj, w))
                    cross_count += 1

        n_intra = sum(len(v) for v in self.intra_edges.values())
        print(f"  KG: {sum(len(v) for v in self.nodes.values())} nodes | "
              f"{n_intra} intra-edges | {len(self.cross_edges)} cross-edges")

    def _get_stacked(self, lid):
        if lid not in self._stacked:
            if lid not in self.nodes or not self.nodes[lid]:
                return None
            self._stacked[lid] = torch.stack([n["emb"] for n in self.nodes[lid]])
        return self._stacked[lid]

    def save(self, path):
        data = {
            "nodes": {str(k): [{"emb": n["emb"].tolist(), "text": n["text"]}
                                for n in v]
                      for k, v in self.nodes.items()},
            "intra_edges": {str(k): v for k, v in self.intra_edges.items()},
            "cross_edges": self.cross_edges,
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  KG saved to {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM):
        kg = cls(emb_dim=emb_dim)
        with open(path) as f:
            data = json.load(f)
        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                kg.nodes[lid].append({"emb": torch.tensor(n["emb"]), "text": n["text"]})
        for k, edges in data["intra_edges"].items():
            kg.intra_edges[int(k)] = [tuple(e) for e in edges]
        kg.cross_edges = [tuple(e) for e in data["cross_edges"]]
        print(f"  KG loaded from {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR  (unchanged)
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS, threshold=UNCERTAINTY_THRESH):
        self.log_C     = math.log(num_classes)
        self.threshold = threshold

    def entropy(self, logits: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=-1)
        eps   = 1e-9
        H     = -(probs * (probs + eps).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits: torch.Tensor) -> torch.Tensor:
        return self.entropy(logits) > self.threshold

    def top_label(self, logits: torch.Tensor) -> torch.Tensor:
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# KG RETRIEVER  (unchanged)
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    def __init__(self, kg: KnowledgeGraph,
                 top_k=KG_TOP_K, top_nodes=KG_TOP_NODES, hop=KG_HOP):
        self.kg        = kg
        self.top_k     = top_k
        self.top_nodes = top_nodes
        self.hop       = hop

    def retrieve(self, h_i: torch.Tensor, rare_ids: list = None,
                 first_pass_label: int = None) -> list:
        h_norm = F.normalize(h_i.unsqueeze(0), dim=-1)

        subgraph_scores = {}
        for lid in self.kg.nodes:
            embs = self.kg._get_stacked(lid)
            if embs is None or embs.shape[0] == 0:
                continue
            embs_norm = F.normalize(embs, dim=-1)
            sims = torch.mv(embs_norm, h_norm.squeeze(0))
            subgraph_scores[lid] = float(sims.max().item())

        sorted_sgs = sorted(subgraph_scores.items(), key=lambda x: -x[1])
        selected   = [lid for lid, _ in sorted_sgs[:self.top_k]]

        results = []

        for lid in selected:
            embs = self.kg._get_stacked(lid)
            if embs is None:
                continue
            embs_norm = F.normalize(embs, dim=-1)
            sims = torch.mv(embs_norm, h_norm.squeeze(0))

            k = min(self.top_nodes, embs.shape[0])
            top_idx  = sims.topk(k).indices.tolist()
            top_sims = sims.topk(k).values.tolist()

            seed_set = set(top_idx)

            if self.hop >= 1:
                for idx in list(seed_set):
                    for (i, j, w) in self.kg.intra_edges.get(lid, []):
                        if i == idx and j not in seed_set:
                            seed_set.add(j)
                        elif j == idx and i not in seed_set:
                            seed_set.add(i)

                for (la, ni, lb, nj, w) in self.kg.cross_edges:
                    if la == lid and ni in seed_set:
                        cross_embs = self.kg._get_stacked(lb)
                        if cross_embs is not None and nj < cross_embs.shape[0]:
                            results.append((cross_embs[nj], w))
                    elif lb == lid and nj in seed_set:
                        cross_embs = self.kg._get_stacked(la)
                        if cross_embs is not None and ni < cross_embs.shape[0]:
                            results.append((cross_embs[ni], w))

            for idx, sim in zip(top_idx, top_sims):
                results.append((embs[idx], float(sim)))

        return results


# ═══════════════════════════════════════════════════════════
# GRAPH ATTENTION FUSION  (unchanged)
# ═══════════════════════════════════════════════════════════
class GraphAttentionFusion(nn.Module):
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT):
        super().__init__()
        self.emb_dim = emb_dim
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, h_i: torch.Tensor,
                neighbours: list, device=None) -> torch.Tensor:
        if not neighbours:
            return torch.zeros_like(h_i)
        if device is None:
            device = h_i.device

        embs    = torch.stack([nb[0] for nb in neighbours]).to(device)
        weights = torch.tensor([nb[1] for nb in neighbours],
                               device=device, dtype=torch.float)

        q = self.proj_q(h_i.unsqueeze(0))
        k = self.proj_k(embs)
        dot   = torch.mv(k, q.squeeze(0)) * self.scale
        alpha = F.softmax(dot * weights, dim=0)
        alpha = self.dropout(alpha)
        v_i   = (alpha.unsqueeze(-1) * embs).sum(dim=0)
        return v_i


# ═══════════════════════════════════════════════════════════
# KG-AUGMENTED MODEL  (extended with SCL on fused embeddings)
# ═══════════════════════════════════════════════════════════
class KGAugmentedModel(nn.Module):
    """
    Extends KGAugmentedModel with [SCL-4]:
    SCL is also applied on fused_sent_vecs during Phase B training,
    so that KG-enriched minority embeddings form tight clusters.
    """
    def __init__(
        self,
        base_model: InLegalBERT_BiLSTM_MHA_CRF,
        kg:         KnowledgeGraph,
        rare_ids:   list,
        retriever:  KGRetriever = None,
    ):
        super().__init__()
        self.base      = base_model
        self.kg        = kg
        self.rare_ids  = rare_ids
        self.retriever = retriever or KGRetriever(kg)
        self.uncertainty = UncertaintyEstimator()

        sent_dim = base_model.sent_out_dim   # 256
        ctx_dim  = base_model.ctx_out_dim    # 128

        self.gat_fusion  = GraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_proj = nn.Sequential(
            nn.Linear(sent_dim * 2, ctx_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100
        )

        # [SCL-4] Projection head on fused embeddings
        self.fused_scl_proj = nn.Sequential(
            nn.Linear(sent_dim, sent_dim),
            nn.GELU(),
            nn.Linear(sent_dim, base_model.scl_proj_dim),
        )

    def _kg_fuse_batch(self, sent_vecs, emissions, lengths, device):
        B, T, sent_dim = sent_vecs.shape
        fused = sent_vecs.clone()

        uncertain_mask = self.uncertainty.is_uncertain(emissions)
        top_labels     = self.uncertainty.top_label(emissions)

        for b in range(B):
            n = int(lengths[b].item())
            for t in range(n):
                uncertain = bool(uncertain_mask[b, t].item())
                pred_lbl  = int(top_labels[b, t].item())
                is_rare   = pred_lbl in self.rare_ids

                if not (uncertain or (RARE_ALWAYS_KG and is_rare)):
                    continue

                h_i = sent_vecs[b, t].detach().cpu()
                neighbours = self.retriever.retrieve(
                    h_i, rare_ids=self.rare_ids, first_pass_label=pred_lbl
                )

                if not neighbours:
                    continue

                v_i = self.gat_fusion(sent_vecs[b, t], neighbours, device=device)
                fused[b, t] = sent_vecs[b, t] + v_i

        return fused

    def _get_fused_scl_embs(
        self,
        fused_sent: torch.Tensor,   # (B, T, sent_dim)
        labels:     torch.Tensor,   # (B, T)
    ):
        """[SCL-4] Project fused sent vecs → normalised contrastive space."""
        B, T, D = fused_sent.shape
        flat_vecs   = fused_sent.reshape(B * T, D)
        flat_labels = labels.reshape(B * T)
        valid_mask  = flat_labels != -100
        flat_vecs   = flat_vecs[valid_mask]
        flat_labels = flat_labels[valid_mask]
        if flat_vecs.shape[0] == 0:
            return None, None
        proj = self.fused_scl_proj(flat_vecs)
        proj = F.normalize(proj, dim=-1)
        return proj, flat_labels

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
        class_weights:  torch.Tensor = None,   # [SCL-4]
        scl_loss_fn:    nn.Module    = None,   # [SCL-4]
        rare_ids:       list         = None,   # [SCL-4]
        scl_sampler:    object       = None,   # [SCL-4]
    ):
        device = input_ids.device

        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )

        fused_sent = self._kg_fuse_batch(
            sent_vecs, base_emissions, lengths, device
        )

        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)

        fused_ctx = self.base.dropout(fused_ctx)

        fused_emissions = self.fusion_classifier(fused_ctx)
        fused_emissions = torch.nan_to_num(fused_emissions, nan=0.0,
                                            posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2], dtype=torch.bool,
                              device=device)

        combined_emissions = (base_emissions + fused_emissions) / 2.0

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            base_crf_loss = -self.base.crf(
                base_emissions, safe_labels, mask=mask, reduction="mean"
            )
            fused_crf_loss = -self.fusion_crf(
                fused_emissions, safe_labels, mask=mask, reduction="mean"
            )
            B2, T2, C = combined_emissions.shape
            ce_loss = self.ce_loss(
                combined_emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
            )

            # [SCL-4] Contrastive loss on fused embeddings
            scl_loss_fused = torch.tensor(0.0, device=device)
            if scl_loss_fn is not None and class_weights is not None:
                proj_fused, flat_labels_fused = self._get_fused_scl_embs(
                    fused_sent, labels
                )
                if proj_fused is not None and proj_fused.shape[0] >= 2:
                    if scl_sampler is not None:
                        proj_fused, flat_labels_fused = scl_sampler.sample(
                            proj_fused, flat_labels_fused
                        )
                    cw = class_weights.to(device)
                    scl_loss_fused = scl_loss_fn(
                        proj_fused, flat_labels_fused, cw, rare_ids=rare_ids
                    )

            loss = ((base_crf_loss + fused_crf_loss) / 2.0
                    + AUX_CE_WEIGHT * ce_loss
                    + SCL_WEIGHT    * scl_loss_fused)   # [SCL-4]

            return loss, combined_emissions, scl_loss_fused.item()
        else:
            decoded = self.fusion_crf.decode(fused_emissions, mask=mask)
            return decoded, fused_emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_trues  = [id2label[x] for x in all_trues]
    str_preds  = [id2label[x] for x in all_preds]
    cls_report = classification_report(
        str_trues, str_preds, labels=LABELS, digits=4, zero_division=0
    )
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


def count_parameters(model):
    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {total_trainable:,} | Frozen: {total_frozen:,}")
    return total_trainable, total_frozen


# ═══════════════════════════════════════════════════════════
# TRAINER  (Phase A: base + SCL)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []
        param_groups.append({
            "params": list(self.model.bert.pooler.parameters()),
            "lr": BERT_LR, "weight_decay": WEIGHT_DECAY,
        })
        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters() if p.requires_grad]
            if params:
                param_groups.append({"params": params, "lr": lr_i,
                                     "weight_decay": WEIGHT_DECAY})
        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf,
            self.model.scl_proj,     # [SCL] include projection head
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        param_groups.append({"params": head_params, "lr": HEAD_LR,
                             "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attn, ttype, labels, lengths in loader:
                input_ids = input_ids.to(self.device)
                attn      = attn.to(self.device)
                ttype     = ttype.to(self.device)
                labels    = labels.to(self.device)
                lengths   = lengths.to(self.device)
                result = self.model(input_ids, attn, ttype,
                                    labels=labels, lengths=lengths)
                loss = result[0]
                if not torch.isnan(loss):
                    total_loss += loss.item(); n += 1
        return total_loss / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attn, ttype, labels, lengths in loader:
                input_ids = input_ids.to(self.device)
                attn      = attn.to(self.device)
                ttype     = ttype.to(self.device)
                lengths   = lengths.to(self.device)
                decoded, _ = self.model(input_ids, attn, ttype,
                                        labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents     = len(all_trues)
            infer_info  = {
                "total_inference_time_s":     total_infer,
                "latency_per_document_ms":    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(
        self,
        train_dataset, dev_dataset, rare_ids,
        tokenizer,
        class_weights:  torch.Tensor = None,   # [SCL]
        scl_loss_fn:    nn.Module    = None,   # [SCL]
        scl_sampler:    object       = None,   # [SCL]
        num_epochs=NUM_EPOCHS_BASE,
    ):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   shuffle=True, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )

        early_stopper = EarlyStopping()
        history = []
        best_f1, best_state = -1.0, None
        total_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, running_scl, n_steps = 0.0, 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                # [SCL-3] Pass SCL components to forward
                result = self.model(
                    ids, attn, ttype,
                    labels=labels, lengths=lengths,
                    class_weights=class_weights,
                    scl_loss_fn=scl_loss_fn,
                    rare_ids=rare_ids,
                    scl_sampler=scl_sampler,
                )
                loss     = result[0]
                scl_val  = result[2] if len(result) > 2 else 0.0

                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                running_scl  += scl_val
                n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            avg_scl_loss   = running_scl  / max(1, n_steps)
            val_loss       = self.compute_val_loss(dev_dataset)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base+SCL] Epoch {epoch:03d}/{num_epochs} | "
                f"loss: {avg_train_loss:.4f} | scl: {avg_scl_loss:.4f} | "
                f"val_loss: {val_loss:.4f} | "
                f"macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s"
            )

            history.append({
                "epoch": epoch, "phase": "base_scl",
                "train_loss": avg_train_loss,
                "scl_loss": avg_scl_loss,
                "val_loss": val_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_micro_f1": val_metrics["micro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "base_scl_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")

        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state or self.model.state_dict(),
                       os.path.join(BEST_MODEL_DIR, "base_scl_model.bin"))
            print(f"  Base+SCL model saved to {BEST_MODEL_DIR}")

        return hist_df, total_time


# ═══════════════════════════════════════════════════════════
# KG BUILDER  (unchanged)
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_knowledge_graph(base_model, train_docs, tokenizer, device=DEVICE):
    print("\n🔨 Building Knowledge Graph from SCL-trained embeddings ...")
    base_model.eval()
    base_model.to(device)
    kg = KnowledgeGraph(emb_dim=base_model.sent_out_dim)
    dummy_dataset = RRCDataset(train_docs, tokenizer)
    loader = DataLoader(dummy_dataset, batch_size=1, shuffle=False,
                        collate_fn=collate_rrc)

    for doc_idx, (ids, attn, ttype, labels, lengths) in enumerate(loader):
        ids     = ids.to(device)
        attn    = attn.to(device)
        ttype   = ttype.to(device)
        lengths = lengths.to(device)
        sent_vecs = base_model.encode_sentences(ids, attn, ttype).squeeze(0)
        n = int(lengths[0].item())
        embs    = sent_vecs[:n].cpu()
        lab_ids = labels[0, :n].tolist()
        kg.add_nodes(embs, lab_ids)
        if (doc_idx + 1) % 50 == 0:
            print(f"  Processed {doc_idx+1} / {len(loader)} docs")

    kg.build_edges()
    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    kg.save(kg_path)
    return kg


# ═══════════════════════════════════════════════════════════
# TRAINER  (Phase B: KG + SCL on fused embeddings)
# ═══════════════════════════════════════════════════════════
class KGTrainer:
    def __init__(self, kg_model: KGAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_params = (
            list(self.model.gat_fusion.parameters())
            + list(self.model.fusion_proj.parameters())
            + list(self.model.fusion_classifier.parameters())
            + list(self.model.fusion_crf.parameters())
            + list(self.model.fused_scl_proj.parameters())   # [SCL-4]
        )
        base_trainable = [p for p in self.model.base.parameters() if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_params,     "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
            {"params": base_trainable, "lr": BERT_LR, "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype,
                                        labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents = len(all_trues)
            infer_info = {
                "total_inference_time_s": total_infer,
                "latency_per_document_ms": total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"kg_inference_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(
        self,
        train_dataset, dev_dataset, rare_ids,
        class_weights:  torch.Tensor = None,   # [SCL-4]
        scl_loss_fn:    nn.Module    = None,   # [SCL-4]
        scl_sampler:    object       = None,   # [SCL-4]
        num_epochs=NUM_EPOCHS_KG,
    ):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   shuffle=True, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )

        early_stopper = EarlyStopping(patience=5)
        history = []
        best_f1, best_state = -1.0, None
        total_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, running_scl, n_steps = 0.0, 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                # [SCL-4] Pass SCL components
                result = self.model(
                    ids, attn, ttype,
                    labels=labels, lengths=lengths,
                    class_weights=class_weights,
                    scl_loss_fn=scl_loss_fn,
                    rare_ids=rare_ids,
                    scl_sampler=scl_sampler,
                )
                loss    = result[0]
                scl_val = result[2] if len(result) > 2 else 0.0

                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item()
                running_scl  += scl_val
                n_steps += 1

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            avg_scl_loss   = running_scl  / max(1, n_steps)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[KG+SCL] Epoch {epoch:02d}/{num_epochs} | "
                f"loss: {avg_train_loss:.4f} | scl: {avg_scl_loss:.4f} | "
                f"macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s"
            )

            history.append({
                "epoch": epoch, "phase": "kg_scl",
                "train_loss": avg_train_loss,
                "scl_loss": avg_scl_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best KG+SCL val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  KG+SCL early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "kg_scl_history.csv"), index=False
        )
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "kg_scl_model.bin"))
            print(f"\n✔ Best KG+SCL model saved (val_macro_f1={best_f1:.4f})")

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION HELPERS
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
        for tick in ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix (KG-RAG+SCL)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    labels = LABELS
    f1s    = [per_class_metrics[l]["f1"] for l in labels]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
              for l in labels]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(labels, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1 (KG-RAG+SCL)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    boundary = len(base_df)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"],
            label="Train Loss", marker="o", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Combined Training Loss"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"],
            label="Val Macro-F1", marker="o", markersize=3)
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"],
            label="Val Rare-F1", marker="s", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Val F1 (Base+SCL → KG+SCL)"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[2]
    if "scl_loss" in all_df.columns:
        ax.plot(all_df["global_epoch"], all_df["scl_loss"].fillna(0),
                label="SCL Loss", marker="^", markersize=3, color="orange")
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("SCL Loss Over Training"); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def print_metrics_table(dev_metrics, test_metrics,
                        base_time=None, kg_time=None,
                        total_trainable=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Macro-Recall",       "macro_recall"),
    ]
    print("\n" + "=" * 72)
    print("FINAL RESULTS — KG-RAG + Supervised Contrastive Learning")
    print("Architecture: InLegalBERT + BiLSTM + MHA + CRF + KG-RAG + SCL")
    print("=" * 72)
    if total_trainable:
        print(f"  Trainable Parameters  : {total_trainable:,}")
    if base_time:
        print(f"  Phase A training time : {base_time/60:.1f} min")
    if kg_time:
        print(f"  Phase B training time : {kg_time/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 72)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 64)
    print(f"  {'Label':<22} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 64)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 64)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT + BiLSTM + MHA + CRF + KG-RAG + SCL")
    print("\nPhase A: Train base model with Supervised Contrastive Learning")
    print("Phase B: Fine-tune KG-Augmented model with SCL on fused embeddings\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    freq_df = pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ])
    freq_df.to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    # ── [SCL] Compute class weights ───────────────────────
    class_weights = compute_class_weights(label_freqs, rare_ids)

    # ── [SCL] Build loss function and sampler ─────────────
    scl_loss_fn = SupConLoss(
        temperature=SCL_TEMPERATURE,
        minority_boost=SCL_MINORITY_BOOST,
    )
    scl_sampler = BalancedContrastiveSampler(
        samples_per_class=SCL_SAMPLES_PER_CLASS,
        rare_ids=rare_ids,
    )

    print("Loading tokenizer ...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ════════════════════════════════════════════════════
    # PHASE A: Base model + SCL
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training with SCL")
    print("=" * 60)
    print(f"  SCL weight:         {SCL_WEIGHT}")
    print(f"  SCL temperature:    {SCL_TEMPERATURE}")
    print(f"  Minority boost:     {SCL_MINORITY_BOOST}x")
    print(f"  Samples per class:  {SCL_SAMPLES_PER_CLASS}")

    base_model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    )

    base_trainer = BaseTrainer(base_model, device=DEVICE)
    base_hist_df, base_time = base_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        tokenizer     = tokenizer,
        class_weights = class_weights,   # [SCL]
        scl_loss_fn   = scl_loss_fn,     # [SCL]
        scl_sampler   = scl_sampler,     # [SCL]
        num_epochs    = NUM_EPOCHS_BASE,
    )

    # ── Build KG from SCL-trained embeddings ──────────────
    print("\n" + "=" * 60)
    print("PHASE A→B: Building Knowledge Graph (SCL-trained embeddings)")
    print("=" * 60)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if os.path.exists(kg_path):
        print("  Found existing KG, loading ...")
        kg = KnowledgeGraph.load(kg_path, emb_dim=base_model.sent_out_dim)
    else:
        kg = build_knowledge_graph(base_model, train_docs, tokenizer, DEVICE)

    # ════════════════════════════════════════════════════
    # PHASE B: KG + SCL on fused embeddings
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: KG-Augmented Fine-Tuning with SCL on Fused Embeddings")
    print("=" * 60)

    retriever = KGRetriever(kg, top_k=KG_TOP_K,
                            top_nodes=KG_TOP_NODES, hop=KG_HOP)
    kg_model = KGAugmentedModel(
        base_model = base_model,
        kg         = kg,
        rare_ids   = rare_ids,
        retriever  = retriever,
    )

    total_trainable, _ = count_parameters(kg_model)

    kg_trainer = KGTrainer(kg_model, device=DEVICE)
    kg_hist_df, kg_time = kg_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        class_weights = class_weights,   # [SCL-4]
        scl_loss_fn   = scl_loss_fn,     # [SCL-4]
        scl_sampler   = scl_sampler,     # [SCL-4]
        num_epochs    = NUM_EPOCHS_KG,
    )

    plot_combined_history(base_hist_df, kg_hist_df)

    # ════════════════════════════════════════════════════
    # EVALUATION
    # ════════════════════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_metrics = kg_trainer.evaluate(dev_dataset, rare_ids,
                                       split_name="dev",
                                       measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + KG-RAG + SCL\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set ...")
    test_metrics = kg_trainer.evaluate(test_dataset, rare_ids,
                                        split_name="test",
                                        measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + KG-RAG + SCL\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pred_df = pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    })
    pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision", "rare_precision",
        "macro_recall", "micro_recall", "weighted_recall", "rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": "InLegalBERT + BiLSTM + MHA + CRF + KG-RAG + SCL",
        "scl_config": {
            "scl_weight": SCL_WEIGHT,
            "temperature": SCL_TEMPERATURE,
            "minority_boost": SCL_MINORITY_BOOST,
            "samples_per_class": SCL_SAMPLES_PER_CLASS,
        },
        "kg_config": {
            "top_k": KG_TOP_K, "top_nodes": KG_TOP_NODES,
            "hop": KG_HOP, "uncertainty_thresh": UNCERTAINTY_THRESH,
            "rare_always_kg": RARE_ALWAYS_KG,
        },
        "timing": {
            "phase_a_s": base_time, "phase_b_s": kg_time,
            "total_s": base_time + kg_time,
        },
        "rare_classes": rare_labels,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        base_time=base_time, kg_time=kg_time,
        total_trainable=total_trainable,
    )
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT + BiLSTM + MHA + CRF + KG-RAG + SCL

Phase A: Train base model with Supervised Contrastive Learning
Phase B: Fine-tune KG-Augmented model with SCL on fused embeddings

Loading JSONL files ...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RARE
   NONE                  4.79%  ( 1377 samples

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-7.
🔥 BERT layers trainable: layers 8-11 + pooler.

[Base+SCL] Epoch 001/60 | loss: 281.7703 | scl: -18.8399 | val_loss: 212.3491 | macro_f1: 0.0391 | rare_f1: 0.0000 | time: 55.4s
  ✔ New best val_macro_f1=0.0391
[Base+SCL] Epoch 002/60 | loss: 228.2590 | scl: -19.1453 | val_loss: 163.1174 | macro_f1: 0.0916 | rare_f1: 0.0000 | time: 55.2s
  ✔ New best val_macro_f1=0.0916
[Base+SCL] Epoch 003/60 | loss: 188.5820 | scl: -19.3326 | val_loss: 123.1727 | macro_f1: 0.2413 | rare_f1: 0.0862 | time: 54.8s
  ✔ New best val_macro_f1=0.2413
[Base+SCL] Epoch 004/60 | loss: 148.1125 | scl: -19.4050 | val_loss: 102.6586 | macro_f1: 0.2649 | rare_f1: 0.1038 | time: 55.4s
  ✔ New best val_macro_f1=0.2649
[Base+SCL] Epoch 005/60 | loss: 125.5207 | scl: -19.4170 | val_loss: 89.8440 | macro_f1: 0.2815 | rare_f1: 0.1188 | time: 55.6s
  ✔ New best val_macro_f1=0.2815
[Base+SCL] Epoch 006/60 | loss: 119.3795 | scl: -19.6349 | val_loss: 81.3958 | macro_f1: 0.32